In [13]:
## Итерация 1 Описание и тестирование
import math
import timeit
import unittest
from typing import Callable

def integrate(f: Callable[[float], float], a: float, b: float, *, n_iter: int = 100000) -> float:
    """
    Вычисляет определенный интеграл функции f на отрезке [a, b] методом левых прямоугольников.

    Аргументы:
        f (Callable[[float], float]): Интегрируемая функция.
        a (float): Нижняя граница интегрирования.
        b (float): Верхняя граница интегрирования.
        n_iter (int, optional): Количество итераций (прямоугольников). По умолчанию 100000.

    Возвращает:
        float: Приближенное значение определенного интеграла.

    Ограничения:
        Метод левых прямоугольников обладает низкой точностью (O(1/n)) по сравнению
        с методами трапеций или Симпсона. Для осциллирующих функций требуется большое n_iter.

    Пример:
        >>> import math
        >>> round(integrate(math.sin, 0, math.pi, n_iter=10000), 2)
        2.0
        >>> round(integrate(lambda x: x**2, 0, 1, n_iter=10000), 2)
        0.33
    """
    acc: float = 0.0
    step: float = (b - a) / n_iter
    for i in range(n_iter):
        acc += f(a + i * step) * step
    return acc

# Проверка doctest
import doctest
doctest.testmod(verbose=True)

class TestIntegrate(unittest.TestCase):
    def test_known_integral_sin(self):
        """Проверка известного интеграла: sin(x) от 0 до pi = 2"""
        result = integrate(math.sin, 0, math.pi, n_iter=100000)
        self.assertAlmostEqual(result, 2.0, places=4)

    def test_stability_to_iterations(self):
        """Проверка устойчивости и сходимости при увеличении n_iter"""
        res_1k = integrate(math.cos, 0, math.pi / 2, n_iter=1000)
        res_100k = integrate(math.cos, 0, math.pi / 2, n_iter=100000)
        exact_value = 1.0

        self.assertLess(abs(res_100k - exact_value), abs(res_1k - exact_value))

        self.assertAlmostEqual(res_100k, exact_value, places=4)

# Запуск тестов
unittest.main(argv=[''], verbosity=2, exit=False)

# Замеры для отчета
print("n_iter=10**4:", timeit.timeit(lambda: integrate(math.cos, 0, math.pi/2, n_iter=10**4), number=10))
print("n_iter=10**5:", timeit.timeit(lambda: integrate(math.cos, 0, math.pi/2, n_iter=10**5), number=10))
print("n_iter=10**6:", timeit.timeit(lambda: integrate(math.cos, 0, math.pi/2, n_iter=10**6), number=10))

test_known_integral_sin (__main__.TestIntegrate.test_known_integral_sin)
Проверка известного интеграла: sin(x) от 0 до pi = 2 ... ok
test_stability_to_iterations (__main__.TestIntegrate.test_stability_to_iterations)
Проверка устойчивости и сходимости при увеличении n_iter ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.035s

OK


Trying:
    import math
Expecting nothing
ok
Trying:
    round(integrate(math.sin, 0, math.pi, n_iter=10000), 2)
Expecting:
    2.0
ok
Trying:
    round(integrate(lambda x: x**2, 0, 1, n_iter=10000), 2)
Expecting:
    0.33
ok
6 items had no tests:
    __main__
    __main__.TestIntegrate
    __main__.TestIntegrate.test_known_integral_sin
    __main__.TestIntegrate.test_stability_to_iterations
    __main__.integrate_async_processes
    __main__.integrate_async_threads
1 items passed all tests:
   3 tests in __main__.integrate
3 tests in 7 items.
3 passed and 0 failed.
Test passed.
n_iter=10**4: 0.013284447999922122
n_iter=10**5: 0.13790406399994026
n_iter=10**6: 1.403776366999864


In [2]:
## Итерация 2 оптимизация с помощью потоков
import concurrent.futures as futures
from functools import partial

def integrate_async_threads(f: Callable[[float], float], a: float, b: float, *, n_jobs: int = 2, n_iter: int = 100000) -> float:
    """Вычисление интеграла с использованием ThreadPoolExecutor."""
    executor = futures.ThreadPoolExecutor(max_workers=n_jobs)

    step_job: float = (b - a) / n_jobs
    iter_per_job: int = n_iter // n_jobs

    spawn = partial(executor.submit, integrate, f, n_iter=iter_per_job)

    fs = [spawn(a + i * step_job, a + (i + 1) * step_job) for i in range(n_jobs)]

    return sum(f.result() for f in futures.as_completed(fs))

# Замеры времени
for jobs in [1, 2, 4, 8]:
    t = timeit.timeit(lambda: integrate_async_threads(math.cos, 0, math.pi/2, n_jobs=jobs, n_iter=10**6), number=5)
    print(f"Threads={jobs}: {t:.4f} sec")

Threads=1: 0.6941 sec
Threads=2: 0.5547 sec
Threads=4: 0.9969 sec
Threads=8: 1.2183 sec


In [3]:
# Итерация 3 оптимизация с помощью процессов
def integrate_async_processes(f: Callable[[float], float], a: float, b: float, *, n_jobs: int = 2, n_iter: int = 100000) -> float:
    """Вычисление интеграла с использованием ProcessPoolExecutor."""
    # для процессов функция f должна быть pickle-совместимой
    # lambda-функции через процессы не передаются
    with futures.ProcessPoolExecutor(max_workers=n_jobs) as executor:
        step_job: float = (b - a) / n_jobs
        iter_per_job: int = n_iter // n_jobs

        spawn = partial(executor.submit, integrate, f, n_iter=iter_per_job)
        fs = [spawn(a + i * step_job, a + (i + 1) * step_job) for i in range(n_jobs)]

        return sum(f.result() for f in futures.as_completed(fs))

# Замеры времени
for jobs in [1, 2, 4, 8]:
    t = timeit.timeit(lambda: integrate_async_processes(math.cos, 0, math.pi/2, n_jobs=jobs, n_iter=10**6), number=5)
    print(f"Processes={jobs}: {t:.4f} sec")

Processes=1: 0.9985 sec
Processes=2: 0.6926 sec
Processes=4: 0.7789 sec
Processes=8: 1.1073 sec


In [10]:
# Итерация 4 Cython
!pip install cython
%load_ext Cython

In [11]:
%cython -a

from libc.math cimport cos

def integrate_cython_optimized(double a, double b, int n_iter=100000):
    """
    Оптимизированная версия Cython.
    """
    cdef double acc = 0.0
    cdef double step = (b - a) / n_iter
    cdef int i
    cdef double x

    for i in range(n_iter):
        x = a + i * step
        acc += cos(x) * step

    return acc

In [12]:
# Итерация 5 noGill
%%cython -a --compile-args=-fopenmp --link-args=-fopenmp
from libc.math cimport cos
from cython.parallel cimport prange

def integrate_cython_nogil(double a, double b, int n_iter=100000, int n_jobs=4):
    cdef double acc = 0.0
    cdef double step = (b - a) / n_iter
    cdef int i
    cdef double x

    # prange автоматически распараллеливает цикл по потокам C-уровня,
    # полностью обходя Python GIL. Cython сам сделает reduction для acc.
    for i in prange(n_iter, nogil=True, num_threads=n_jobs):
        x = a + i * step
        acc += cos(x) * step
    return acc